# 03 Model 1: Word Embeddings + Classifier
Training of architecture 1: frozen pretrained word embeddings with a 2-layer classifier (ReLU non-linearity). Only the classifier is trained. Experiments are tracked with Weights & Biases.

Load the features and labels that were saved in `02_preprocessing.ipynb`. The test set is not loaded here: it is only used once at the end, in `05_evaluation.ipynb`.

In [1]:
import numpy as np
import torch

data = np.load("features/model1.npz")
X_train = torch.tensor(data["X_train"])
y_train = torch.tensor(data["y_train"])
X_valid = torch.tensor(data["X_valid"])
y_valid = torch.tensor(data["y_valid"])
print(X_train.shape, X_train.dtype)
print(y_train.shape, y_train.dtype)

torch.Size([16801, 1200]) torch.float32
torch.Size([16801]) torch.int64


## Classifier
The 2-layer classifier like in the course exercise "PyTorch" (Week 1): linear layer, ReLU, linear layer. Input: 1200 features, output: 2 classes (h1 or h2).

In [2]:
import torch.nn as nn

class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

model = Classifier(1200, 2, inner_dim=100)
print(model)

Classifier(
  (linear1): Linear(in_features=1200, out_features=100, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=100, out_features=2, bias=True)
)


Number of parameters per layer (weights and biases), as in the course exercise "PyTorch" (Week 1).

In [3]:
for name, params in model.named_parameters():
    print(name, params.shape)

linear1.weight torch.Size([100, 1200])
linear1.bias torch.Size([100])
linear2.weight torch.Size([2, 100])
linear2.bias torch.Size([2])


Test the untrained classifier on the first 5 training examples: one pair of scores (h1, h2) per example.

In [4]:
x = X_train[:5]
output = model(x)
print(x.shape)
print(output.shape)
print(output)

torch.Size([5, 1200])
torch.Size([5, 2])
tensor([[0.0501, 0.0319],
        [0.0602, 0.0048],
        [0.0524, 0.0280],
        [0.0680, 0.0197],
        [0.0462, 0.0251]], grad_fn=<AddmmBackward0>)


## Loss
Cross-entropy loss for the 2 classes (course slides "Preliminaries", slide 27). `nn.CrossEntropyLoss` applies the softmax to the raw scores itself, so the classifier has no softmax layer. Before training, the loss should be about -log(0.5) ≈ 0.69, the value of random guessing between 2 classes.

In [5]:
criterion = nn.CrossEntropyLoss()

y = y_train[:5]
loss = criterion(output, y)
print(y)
print(loss)

tensor([1, 1, 0, 0, 1])
tensor(0.6955, grad_fn=<NllLossBackward0>)


## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Loading the features: `np.load` as in the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.savez.html.
- Tensors: `torch.tensor` as in the course exercise "PyTorch" (Week 1).
- Naming: `X` (features, uppercase because it is a matrix) and `y` (labels) following the scikit-learn convention, https://scikit-learn.org/stable/glossary.html.
- Classifier: class `MyNetwork` from the course exercise "PyTorch" (Week 1), renamed to `Classifier`, based on the PyTorch tutorial "Neural Networks", https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html; architecture "linear layer → ReLU → linear layer" from the course slides "Preliminaries" (Week 1), slide 25.
- Parameters and test forward pass: `named_parameters()` and a forward pass with a small batch as in the course exercise "PyTorch" (Week 1).
- Loss: cross-entropy from the course slides "Preliminaries" (Week 1), slide 27; `criterion` and `loss = criterion(output, target)` as in the course exercise "PyTorch" (Week 1); `nn.CrossEntropyLoss` (applies the softmax to the raw scores) from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html.
- Tools: code written with the help of Claude (Anthropic); all code reviewed and understood by me.